# Drill Core Image Classification — Data Pipeline
**Google Colab** | Run cells top to bottom

## Cell 1 — Install Dependencies
Run once. If Colab prompts a runtime restart, do it, then skip this cell.

In [ ]:
!pip install timm huggingface_hub --quiet

## Cell 2 — Imports

In [ ]:
import os
import random
import warnings
import numpy as np
import torch
import torch.nn as nn
import torchvision
from torchvision import transforms, datasets
from torch.utils.data import DataLoader, random_split
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from collections import Counter

warnings.filterwarnings('ignore')
print(f"PyTorch version: {torch.__version__}")
print(f"Torchvision version: {torchvision.__version__}")

## Cell 3 — Config
All hyperparameters and paths live here. Change things **only** in this cell.

> **Tip:** If Google Drive mount fails (Cell 5), set `CKPT_DIR = '/content/checkpoints'` here.

In [ ]:
class Config:
    # --- Paths (Colab) ---
    DATA_ROOT   = '/content/DCID/DCID-512-7'
    TRAIN_DIR   = os.path.join(DATA_ROOT, 'train')
    TEST_DIR    = os.path.join(DATA_ROOT, 'test')
    # Primary: save to Drive so checkpoints survive session resets.
    # Fallback (used automatically in Cell 5 if Drive mount fails):
    CKPT_DIR    = '/content/drive/MyDrive/drill_core_project/checkpoints'
    CKPT_FALLBACK = '/content/checkpoints'

    # --- Data ---
    IMG_SIZE    = 224       # Standard ImageNet input size
    NUM_CLASSES = 7
    VAL_SPLIT   = 0.1      # 10% of training data -> validation
    SEED        = 42

    # --- Training ---
    BATCH_SIZE  = 32
    NUM_WORKERS = 2
    LR          = 1e-3
    EPOCHS      = 20
    BACKBONE    = 'efficientnet_b0'

cfg = Config()
print("Config loaded.")

## Cell 4 — Reproducibility + Device

In [ ]:
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(cfg.SEED)

device = torch.device(
    'cuda' if torch.cuda.is_available() else
    'mps'  if torch.backends.mps.is_available() else
    'cpu'
)
print(f"Device: {device}")
if device.type == 'cuda':
    print(f"GPU:   {torch.cuda.get_device_name(0)}")
    print(f"VRAM:  {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️  No GPU detected — training will be very slow. Enable GPU in Runtime → Change runtime type.")

## Cell 5 — Mount Google Drive
Saves checkpoints so your work survives Colab session resets.

> If the auth popup doesn't appear, enable third-party cookies in Chrome, then re-run this cell.
> If mounting fails entirely, the cell automatically falls back to `/content/checkpoints`.

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs(cfg.CKPT_DIR, exist_ok=True)
    print(f"✅ Drive mounted. Checkpoints: {cfg.CKPT_DIR}")
except Exception as e:
    print(f"⚠️  Drive mount failed ({e}). Using local fallback.")
    cfg.CKPT_DIR = cfg.CKPT_FALLBACK
    os.makedirs(cfg.CKPT_DIR, exist_ok=True)
    print(f"Checkpoints will be saved to: {cfg.CKPT_DIR}")
    print("Note: these will be lost when the Colab session ends.")

## Cell 6 — Download DCID Dataset
**Source:** [168sir/drill-core-image-dataset](https://huggingface.co/datasets/168sir/drill-core-image-dataset) · License: CC BY-NC 4.0

⏱ ~5–10 min on Colab free tier. Run **once per session** (data doesn't persist across resets).

After unzipping:
```
/content/DCID/DCID-512-7/
    train/<class>/   ← 4,000 images per class
    test/<class>/    ← 1,000 images per class
```

In [ ]:
from huggingface_hub import hf_hub_download
import zipfile, shutil

# Skip download if data already exists (re-run safety)
if os.path.isdir(cfg.DATA_ROOT):
    print(f"✅ Dataset already present at {cfg.DATA_ROOT} — skipping download.")
else:
    print("Downloading DCID.zip (~3.82 GB) — please wait...")
    zip_path = hf_hub_download(
        repo_id   = "168sir/drill-core-image-dataset",
        filename  = "DCID.zip",
        repo_type = "dataset",
        local_dir = "/content"
    )
    print(f"Downloaded to: {zip_path}")

    # The zip also contains DCID-512-35, noise-512-7, noise-512-35 variants.
    # One entry in DCID-512-35 has a bad CRC-32, so extractall() crashes.
    # Fix: extract ONLY the DCID-512-7 subtree we actually need.
    TARGET_PREFIX = "DCID/DCID-512-7/"
    print(f"Extracting '{TARGET_PREFIX}' only (skips other variants that have CRC errors)...")
    with zipfile.ZipFile(zip_path, 'r') as z:
        members = [m for m in z.namelist() if m.startswith(TARGET_PREFIX)]
        if not members:
            raise RuntimeError(
                f"No entries found under '{TARGET_PREFIX}' — archive layout may have changed."
            )
        print(f"  → {len(members):,} entries to extract...")
        for member in members:
            z.extract(member, "/content/DCID_raw")

    # z.extract preserves the full internal path, so we get:
    #   /content/DCID_raw/DCID/DCID-512-7/  →  move to  /content/DCID/DCID-512-7/
    nested = "/content/DCID_raw/DCID/DCID-512-7"
    final  = "/content/DCID/DCID-512-7"
    os.makedirs("/content/DCID", exist_ok=True)
    if os.path.isdir(nested):
        shutil.move(nested, final)
    else:
        # fallback: archive root was already flat
        shutil.move(f"/content/DCID_raw/{TARGET_PREFIX.rstrip('/')}", final)
    shutil.rmtree("/content/DCID_raw", ignore_errors=True)
    print(f"✅ Done! Dataset ready at {cfg.DATA_ROOT}")

## Cell 7 — Verify Dataset Structure

In [ ]:
def verify_dataset(train_dir, test_dir):
    assert os.path.isdir(train_dir), f"Train dir not found: {train_dir}"
    assert os.path.isdir(test_dir),  f"Test dir not found:  {test_dir}"
    # Filter out hidden files (e.g. .DS_Store on macOS-created zips)
    train_classes = sorted(c for c in os.listdir(train_dir) if not c.startswith('.'))
    test_classes  = sorted(c for c in os.listdir(test_dir)  if not c.startswith('.'))
    assert train_classes == test_classes, f"Train/test classes don't match!\n  Train: {train_classes}\n  Test:  {test_classes}"
    print(f"\n✅ Dataset verified — {len(train_classes)} classes:")
    for cls in train_classes:
        n_train = len([f for f in os.listdir(os.path.join(train_dir, cls)) if not f.startswith('.')])
        n_test  = len([f for f in os.listdir(os.path.join(test_dir,  cls)) if not f.startswith('.')])
        print(f"  {cls:35s}  train: {n_train:5d}  test: {n_test:5d}")

verify_dataset(cfg.TRAIN_DIR, cfg.TEST_DIR)

## Cell 8 — Transforms
- **Training:** augmentation (random crop, flips, rotation, color jitter) to simulate real-world variation in drill core photos
- **Val / Test:** only resize + normalize — deterministic, no randomness

In [ ]:
# ImageNet normalization stats — backbone was pretrained on ImageNet
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transforms = transforms.Compose([
    transforms.Resize((256, 256)),              # slightly larger than target
    transforms.RandomCrop(cfg.IMG_SIZE),        # random 224x224 crop
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1, hue=0.05),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

eval_transforms = transforms.Compose([
    transforms.Resize((cfg.IMG_SIZE, cfg.IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

print("Transforms defined.")

## Cell 9 — Dataset + Train/Val Split
`SubsetWithTransform` lets the train and val subsets have **different transforms** even though they share the same base `ImageFolder`. Without it, both would get the same augmentations (PyTorch's `random_split` doesn't clone the transform).

In [ ]:
class SubsetWithTransform(torch.utils.data.Dataset):
    """Applies a per-subset transform to a torch Subset."""
    def __init__(self, subset, transform):
        self.subset    = subset
        self.transform = transform

    def __getitem__(self, idx):
        img, label = self.subset[idx]   # returns PIL image (transform=None on base)
        if self.transform:
            img = self.transform(img)
        return img, label

    def __len__(self):
        return len(self.subset)


# Load raw training images with no transform so random_split works cleanly
full_train_raw = datasets.ImageFolder(root=cfg.TRAIN_DIR, transform=None)
test_dataset   = datasets.ImageFolder(root=cfg.TEST_DIR,  transform=eval_transforms)

CLASS_NAMES = full_train_raw.classes
NUM_CLASSES = len(CLASS_NAMES)
print(f"Train images: {len(full_train_raw)}")
print(f"Test images:  {len(test_dataset)}")
print(f"Classes ({NUM_CLASSES}): {CLASS_NAMES}")

# 90 / 10 split
val_size   = int(cfg.VAL_SPLIT * len(full_train_raw))
train_size = len(full_train_raw) - val_size

train_subset, val_subset = random_split(
    full_train_raw,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(cfg.SEED)
)

train_dataset = SubsetWithTransform(train_subset, train_transforms)
val_dataset   = SubsetWithTransform(val_subset,   eval_transforms)

print(f"\nSplit  →  Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")

## Cell 10 — DataLoaders
`pin_memory=True` keeps batches in page-locked CPU memory for faster CPU→GPU transfer (only meaningful when a GPU is present).

In [ ]:
use_pin_memory = (device.type == 'cuda')

train_loader = DataLoader(
    train_dataset, batch_size=cfg.BATCH_SIZE,
    shuffle=True,  num_workers=cfg.NUM_WORKERS, pin_memory=use_pin_memory
)
val_loader = DataLoader(
    val_dataset, batch_size=cfg.BATCH_SIZE,
    shuffle=False, num_workers=cfg.NUM_WORKERS, pin_memory=use_pin_memory
)
test_loader = DataLoader(
    test_dataset, batch_size=cfg.BATCH_SIZE,
    shuffle=False, num_workers=cfg.NUM_WORKERS, pin_memory=use_pin_memory
)

print(f"Batches  →  Train: {len(train_loader)} | Val: {len(val_loader)} | Test: {len(test_loader)}")

# Quick shape sanity check
imgs, labels = next(iter(train_loader))
print(f"Batch shape: {imgs.shape}   (batch × channels × H × W)")
print(f"Labels sample: {labels[:8].tolist()}")

## Cell 11 — Visualize Sample Images
Always inspect augmented images before training — verify they look sensible.

In [ ]:
def denormalize(tensor):
    """Reverse ImageNet normalization for display."""
    img = tensor.numpy().transpose(1, 2, 0)
    img = np.array(IMAGENET_STD) * img + np.array(IMAGENET_MEAN)
    return np.clip(img, 0, 1)


def sample_one_per_class(dataset, class_names):
    """
    Efficiently grab one image per class using the underlying dataset's
    class-to-index mapping instead of iterating through the whole dataset.
    """
    # Build a mapping: class_idx -> first matching global index
    base_ds = dataset.subset.dataset          # the raw ImageFolder
    indices = dataset.subset.indices          # indices into base_ds
    class_to_sample = {}
    for pos, global_idx in enumerate(indices):
        label = base_ds.targets[global_idx]
        if label not in class_to_sample:
            class_to_sample[label] = pos      # position within the Subset
        if len(class_to_sample) == len(class_names):
            break
    return class_to_sample


class_to_pos = sample_one_per_class(train_dataset, CLASS_NAMES)

n_cols = 4
n_rows = (len(CLASS_NAMES) + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 4, n_rows * 4))
axes = axes.flatten()

for i, label in enumerate(sorted(class_to_pos)):
    img, _ = train_dataset[class_to_pos[label]]
    axes[i].imshow(denormalize(img))
    axes[i].set_title(CLASS_NAMES[label], fontsize=11, fontweight='bold')
    axes[i].axis('off')
for j in range(i + 1, len(axes)):
    axes[j].axis('off')

plt.suptitle('One sample per class — Training set (with augmentation)', fontsize=14)
plt.tight_layout()
plt.savefig('sample_images.png', dpi=120, bbox_inches='tight')
plt.show()
print("Saved: sample_images.png")

## Cell 12 — Visualize Class Distribution
Verify the dataset is balanced across train, val, and test.

In [ ]:
def get_labels(ds):
    """Extract label list from an ImageFolder or SubsetWithTransform."""
    if hasattr(ds, 'targets'):           # plain ImageFolder
        return ds.targets
    # SubsetWithTransform wrapping a Subset
    inner = ds.subset                    # torch.utils.data.Subset
    return [inner.dataset.targets[i] for i in inner.indices]


train_labels = get_labels(train_dataset)
val_labels   = get_labels(val_dataset)
test_labels  = test_dataset.targets

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
palette = sns.color_palette("Blues_d", len(CLASS_NAMES))

for ax, labels, title in zip(
        axes,
        [train_labels, val_labels, test_labels],
        ['Train', 'Val', 'Test']):
    counts = Counter(labels)
    bars = ax.bar(
        [CLASS_NAMES[i] for i in sorted(counts)],
        [counts[i]      for i in sorted(counts)],
        color=palette
    )
    ax.bar_label(bars, fontsize=9)
    ax.set_title(f'{title} Distribution  (n={sum(counts.values())})',
                 fontsize=12, fontweight='bold')
    ax.tick_params(axis='x', rotation=45)
    ax.set_ylabel('Image count')
    ax.set_ylim(0, max(counts.values()) * 1.2)
    sns.despine(ax=ax)

plt.tight_layout()
plt.savefig('class_distribution.png', dpi=120, bbox_inches='tight')
plt.show()
print("Saved: class_distribution.png")
print("\n✅ Data pipeline complete — ready to build the model.")